# Phase Advance test

In [1]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import RF_Track as rft
from pathlib import Path
project_root_path = Path.cwd().resolve()
while not (project_root_path / "Interfaces").exists() and project_root_path.parent != project_root_path:
    project_root_path = project_root_path.parent
sys.path.insert(0, str(project_root_path))
os.chdir(project_root_path)
from Interfaces.CLEAR.InterfaceCLEAR_RFTrack import InterfaceCLEAR_RFTrack
from Interfaces.ATF2.InterfaceATF2_Ext_RFTrack import InterfaceATF2_Ext_RFTrack


RF-Track, version 2.6.3

Copyright (C) 2016-2026 CERN, Geneva, Switzerland. All rights reserved.

Author and contact:
 Andrea Latina <andrea.latina@cern.ch>
 BE-ABP Group
 CERN
 CH-1211 GENEVA 23
 SWITZERLAND

This software is distributed under a CERN proprietary software
license in the hope that it will be useful, but WITHOUT ANY WARRANTY;
not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

See the COPYRIGHT and LICENSE files at the top-level directory of
the RF-Track download area: https://gitlab.cern.ch/rf-track

RF-Track was compiled with GSL-2.8 and fftw-3.3.11



[RF-Track] Could not check for updates.


We specify initial Twiss parameters, just to check the mathemtatical equations.

In [2]:
emittance = 2.0
beta = 3.0
alpha = -1.0
gamma=(1+alpha**2)/beta

We know, that the shape of the beam in phase space can be specified by:
$$
\Sigma =
\varepsilon
\begin{pmatrix}
\beta & -\alpha \\
-\alpha & \gamma
\end{pmatrix}
$$

In [3]:
beam_matrix = emittance * np.array([[beta, -alpha],
                                    [-alpha, gamma]])

$$
\varepsilon = \sqrt{\det(\Sigma)}
$$

In [4]:
emittance_from_beam_matrix = np.sqrt(np.linalg.det(beam_matrix))
print(emittance_from_beam_matrix)

2.0


In [5]:
nparticles = 200000

We want a centered beam, so we want an avg=0

In [6]:
particles = np.random.multivariate_normal(mean = [0,0], cov = beam_matrix, size = nparticles) # x | x'
                                                                                              # x | x'
x= particles[:,0]
xp= particles[:,1]


$$
\Sigma =
\begin{pmatrix}
\operatorname{Var}(x) & \operatorname{Cov}(x,x') \\
\operatorname{Cov}(x,x') & \operatorname{Var}(x')
\end{pmatrix}
$$

$$
\varepsilon = \sqrt{\det(\Sigma)}
$$

$$
\beta = \frac{\Sigma_{11}}{\varepsilon}
$$

$$
\alpha = -\frac{\Sigma_{12}}{\varepsilon}
$$

In [7]:
covariance_matrix = np.cov(x,xp)
emittance_from_cov_matrix = np.sqrt(np.linalg.det(covariance_matrix))
beta_from_cov_matrix = covariance_matrix[0][0]/emittance_from_cov_matrix
alpha_from_cov_matrix = -covariance_matrix[0][1]/emittance_from_cov_matrix

In [8]:
print(emittance_from_cov_matrix)
print(beta_from_cov_matrix)
print(alpha_from_cov_matrix)

2.0053160735587885
3.0000400759882497
-1.0009680866376296


The result is basically identical with initial Twiss parameters, so we confirmed the equations used.

$$
R_{\mathrm{rot}} =
\begin{pmatrix}
\cos\mu & \sin\mu \\
-\sin\mu & \cos\mu
\end{pmatrix}
$$

In [9]:
mu = 90
mu = np.deg2rad(mu)
R_rot = np.array([[np.cos(mu), np.sin(mu)],
                 [-np.sin(mu), np.cos(mu)]])


Transport phase space by phase advance 90 degrees.

$$
\mathbf{X}_0 =
\begin{pmatrix}
x \\
x'
\end{pmatrix}
$$

$$
\mathbf{X}_1 = R_{\mathrm{rot}}\,\mathbf{X}_0
$$

$$
\begin{pmatrix}
x_1 \\
x_1'
\end{pmatrix}
=
\begin{pmatrix}
\cos\mu & \sin\mu \\
-\sin\mu & \cos\mu
\end{pmatrix}
\begin{pmatrix}
x \\
x'
\end{pmatrix}
$$

$$
x_1 = x\cos\mu + x'\sin\mu
$$

$$
x_1' = -x\sin\mu + x'\cos\mu
$$

In [10]:
X0_test = particles.T
X1 = R_rot @ X0_test
x1 = X1[0, :]
xp1 = X1[1, :]

In [11]:
covariance_matrix_after_rotation = np.cov(x1,xp1)
emittance_after_rotation = np.sqrt(np.linalg.det(covariance_matrix_after_rotation))
print("Emittance before rotation: ", emittance_from_cov_matrix)
print("Emittance after rotation: ", emittance_after_rotation)

Emittance before rotation:  2.0053160735587885
Emittance after rotation:  2.005316073558788


For each phase advance $\mu$, the phase-space distribution is transformed as

$$
\begin{pmatrix}
x_\mu \\
x'_\mu
\end{pmatrix}
=
R(\mu)
\begin{pmatrix}
x \\
x'
\end{pmatrix}.
$$

The beam size at the screen is obtained from the variance of the
projected $x$ distribution:

$$
\sigma_x^2(\mu) = \operatorname{Var}(x_\mu).
$$

The beam sizes are evaluated at three phase advances:

$$
\sigma_0^2 = \sigma_x^2(0^\circ), \qquad
\sigma_{45}^2 = \sigma_x^2(45^\circ), \qquad
\sigma_{90}^2 = \sigma_x^2(90^\circ).
$$

In [12]:
mus = [0, 45, 90]
sigma2_values = []

for mu in mus:
    mu = np.deg2rad(mu)
    R_rot = np.array([[np.cos(mu), np.sin(mu)],
                      [-np.sin(mu), np.cos(mu)]])
    Xi = R_rot @ X0_test
    x_screen = Xi[0, :]
    sigma2 = np.var(x_screen)
    sigma2_values.append(sigma2)

sigma0_2 = sigma2_values[0]
sigma45_2 = sigma2_values[1]
sigma90_2 = sigma2_values[2]

After the phase-space rotation,

$$
x_\mu = x\cos\mu + x'\sin\mu.
$$

Therefore, the beam size at the screen is

$$
\sigma_\mu^2
=
\Sigma_{11}\cos^2\mu
+2\Sigma_{12}\cos\mu\sin\mu
+\Sigma_{22}\sin^2\mu.
$$

$$
\mu = 0^\circ:
\qquad
\sigma_0^2 = \Sigma_{11}
$$

$$
\mu = 45^\circ:
\qquad
\sigma_{45}^2 =
\frac{1}{2}\Sigma_{11}
+ \Sigma_{12}
+ \frac{1}{2}\Sigma_{22}
$$

$$
\Sigma_{12}
=
\sigma_{45}^2
-\frac{1}{2}\left(\sigma_0^2+\sigma_{90}^2\right)
$$

$$
\mu = 90^\circ:
\qquad
\sigma_{90}^2 = \Sigma_{22}
$$

In [13]:
Sigma11 = sigma0_2
Sigma12 = sigma45_2 - 0.5 * (sigma0_2 + sigma90_2)
Sigma22 = sigma90_2

In [14]:
reconstructed_emittance_matrix = np.array([
    [Sigma11, Sigma12],
    [Sigma12, Sigma22]
])
reconstructed_emittance = np.sqrt(np.linalg.det(reconstructed_emittance_matrix))
print(reconstructed_emittance)

2.005306046978409


Definition of a function that implements rotation matrix R and allows us to give certain values of phase advances.

In [15]:
def R(mu_degrees):
    mu = np.deg2rad(mu_degrees)
    R_rot = np.array([[np.cos(mu), np.sin(mu)],
                      [-np.sin(mu), np.cos(mu)]])
    return R_rot

Picking three "screens".

In [16]:
mus = [0, 45, 90]
sigma2_values = []
for mu in mus:
    R_rot = R(mu)
    Xi = R_rot @ X0_test
    x_screen = Xi[0, :]
    xp_screen = Xi[1, :]
    sigma2 = np.var(x_screen)
    sigma2_values.append(sigma2)
    print(mu, np.var(x_screen))

0 6.01599850555682
45 5.684320437361981
90 1.3381476552334206


$$
\begin{pmatrix}
\sigma_1^2 \\
\sigma_2^2 \\
\sigma_3^2
\end{pmatrix}
=
\begin{pmatrix}
\cos^2\mu_1 & 2\cos\mu_1\sin\mu_1 & \sin^2\mu_1 \\
\cos^2\mu_2 & 2\cos\mu_2\sin\mu_2 & \sin^2\mu_2 \\
\cos^2\mu_3 & 2\cos\mu_3\sin\mu_3 & \sin^2\mu_3
\end{pmatrix}
\begin{pmatrix}
\Sigma_{11} \\
\Sigma_{12} \\
\Sigma_{22}
\end{pmatrix}.
$$

In [17]:
mu1 = np.deg2rad(mus[0])
mu2 = np.deg2rad(mus[1])
mu3 = np.deg2rad(mus[2])

A = np.array([
    [np.cos(mu1)**2, 2 * np.cos(mu1) * np.sin(mu1), np.sin(mu1)**2],
    [np.cos(mu2)**2, 2 * np.cos(mu2) * np.sin(mu2), np.sin(mu2)**2],
    [np.cos(mu3)**2, 2 * np.cos(mu3) * np.sin(mu3), np.sin(mu3)**2],
])

# A * x = b
# x = A^(-1)b
# x = np.linalg.solve(A, b)

Sigma_elements = np.linalg.solve(A, sigma2_values)
Sigma11 = Sigma_elements[0]
Sigma12 = Sigma_elements[1]
Sigma22 = Sigma_elements[2]

Sigma_matrix = np.array([
    [Sigma11, Sigma12],
    [Sigma12, Sigma22]
])


In [18]:
emittance_reconstructed = np.sqrt(np.linalg.det(Sigma_matrix))
print(emittance_reconstructed)
beta_reconstructed = Sigma11 / emittance_reconstructed
print(beta_reconstructed)
alpha_reconstructed = -Sigma12 / emittance_reconstructed
print(alpha_reconstructed)

2.00530604697841
3.000040075988257
-1.0009680866376356


A test of different phase advances. We want to see when the reconstruction stops to be so precise.

In [19]:
def R(mu_degrees):
    mu = np.deg2rad(mu_degrees)
    R_rot = np.array([[np.cos(mu), np.sin(mu)],
                      [-np.sin(mu), np.cos(mu)]])
    return R_rot

def obtain_sigma2_values(X0, mus):
    sigma2_values = []
    for mu in mus:
        R_rot = R(mu)
        Xi = R_rot @ X0
        x_screen = Xi[0, :]
        xp_screen = Xi[1, :]
        sigma2 = np.var(x_screen)
        sigma2_values.append(sigma2)
    return sigma2_values

def reconstruct_Sigma(mus, sigma2_values):
    mu1 = np.deg2rad(mus[0])
    mu2 = np.deg2rad(mus[1])
    mu3 = np.deg2rad(mus[2])

    A = np.array([
        [np.cos(mu1)**2, 2 * np.cos(mu1) * np.sin(mu1), np.sin(mu1)**2],
        [np.cos(mu2)**2, 2 * np.cos(mu2) * np.sin(mu2), np.sin(mu2)**2],
        [np.cos(mu3)**2, 2 * np.cos(mu3) * np.sin(mu3), np.sin(mu3)**2],
    ])

    Sigma_elements = np.linalg.solve(A, sigma2_values)
    Sigma11 = Sigma_elements[0]
    Sigma12 = Sigma_elements[1]
    Sigma22 = Sigma_elements[2]

    Sigma_matrix = np.array([
        [Sigma11, Sigma12],
        [Sigma12, Sigma22]
    ])

    emittance_reconstructed = np.sqrt(np.linalg.det(Sigma_matrix))
    beta_reconstructed = Sigma11 / emittance_reconstructed
    alpha_reconstructed = -Sigma12 / emittance_reconstructed
    return emittance_reconstructed, beta_reconstructed, alpha_reconstructed


def relative_errors(emittance_reconstructed, beta_reconstructed, alpha_reconstructed):
    relative_emittance_error = abs(emittance_reconstructed- emittance)/emittance *100
    relative_beta_error = abs(beta_reconstructed- beta)/beta *100
    relative_alpha_error = abs(alpha_reconstructed- alpha)/abs(alpha) *100
    return relative_emittance_error, relative_beta_error, relative_alpha_error

First, let's try without any errors.

In [20]:
mus = [0 , 45, 90]
sigma2_values = obtain_sigma2_values(X0_test, mus)
emittance_reconstructed, beta_reconstructed, alpha_reconstructed = reconstruct_Sigma(mus, sigma2_values)
print(emittance_reconstructed)
print(beta_reconstructed)
print(alpha_reconstructed)

2.00530604697841
3.000040075988257
-1.0009680866376356


In [21]:
mus = [0, 35, 70]
sigma2_values = obtain_sigma2_values(X0_test, mus)
emittance_reconstructed, beta_reconstructed, alpha_reconstructed = reconstruct_Sigma(mus, sigma2_values)
print(emittance_reconstructed)
print(beta_reconstructed)
print(alpha_reconstructed)

2.005306046978406
3.0000400759882626
-1.000968086637639


In [22]:
mus = [0, 30, 60]
sigma2_values = obtain_sigma2_values(X0_test, mus)
emittance_reconstructed, beta_reconstructed, alpha_reconstructed = reconstruct_Sigma(mus, sigma2_values)
print(emittance_reconstructed)
print(beta_reconstructed)
print(alpha_reconstructed)

2.005306046978412
3.000040075988254
-1.0009680866376345


In [23]:
mus = [0, 20, 40]
sigma2_values = obtain_sigma2_values(X0_test, mus)
emittance_reconstructed, beta_reconstructed, alpha_reconstructed = reconstruct_Sigma(mus, sigma2_values)
print(emittance_reconstructed)
print(beta_reconstructed)
print(alpha_reconstructed)

2.005306046978405
3.000040075988264
-1.0009680866376391


In [24]:
mus = [0, 5, 10]
sigma2_values = obtain_sigma2_values(X0_test, mus)
emittance_reconstructed, beta_reconstructed, alpha_reconstructed = reconstruct_Sigma(mus, sigma2_values)
print(emittance_reconstructed)
print(beta_reconstructed)
print(alpha_reconstructed)

2.0053060469783084
3.0000400759884087
-1.0009680866376902


I introduce some errors and try the same reconstruction. Without the errors, reconstruction still works.

In [25]:
noise_level = 0.01
rng = np.random.default_rng(42)
noise = rng.normal(0, noise_level, size=3)

In [26]:
mus = [0, 45, 90]
sigma2_values = obtain_sigma2_values(X0_test, mus)
sigma2_values = sigma2_values * (1+noise)
emittance_reconstructed, beta_reconstructed, alpha_reconstructed = reconstruct_Sigma(mus, sigma2_values)
relative_emittance_error, relative_beta_error, relative_alpha_error = relative_errors(emittance_reconstructed, beta_reconstructed, alpha_reconstructed)
print(f"Reconstructed emittance: {emittance_reconstructed}, initial emittance: {emittance}, relative emittance error: {relative_emittance_error} %")
print(f"Reconstructed beta: {beta_reconstructed}, initial beta: {beta}, relative beta error: {relative_beta_error} %")
print(f"Reconstructed alpha: {alpha_reconstructed}, initial alpha: {alpha}, relative alpha error: {relative_alpha_error} %")

Reconstructed emittance: 2.096492718325322, initial emittance: 2.0, relative emittance error: 4.824635916266096 %
Reconstructed beta: 2.87829775308795, initial beta: 3.0, relative beta error: 4.056741563734997 %
Reconstructed alpha: -0.9224665322916408, initial alpha: -1.0, relative alpha error: 7.753346770835923 %


In [27]:
mus = [0, 35, 70]
sigma2_values = obtain_sigma2_values(X0_test, mus)
sigma2_values = sigma2_values * (1+noise)
emittance_reconstructed, beta_reconstructed, alpha_reconstructed = reconstruct_Sigma(mus, sigma2_values)
relative_emittance_error, relative_beta_error, relative_alpha_error = relative_errors(emittance_reconstructed, beta_reconstructed, alpha_reconstructed)
print(f"Reconstructed emittance: {emittance_reconstructed}, initial emittance: {emittance}, relative emittance error: {relative_emittance_error} %")
print(f"Reconstructed beta: {beta_reconstructed}, initial emittance: {beta}, relative emittance error: {relative_beta_error} %")
print(f"Reconstructed alpha: {alpha_reconstructed}, initial emittance: {alpha}, relative emittance error: {relative_alpha_error} %")

Reconstructed emittance: 2.284286372947595, initial emittance: 2.0, relative emittance error: 14.214318647379741 %
Reconstructed beta: 2.6416697801048694, initial emittance: 3.0, relative emittance error: 11.94434066317102 %
Reconstructed alpha: -0.8246035921453033, initial emittance: -1.0, relative emittance error: 17.53964078546967 %


In [28]:
mus = [0, 30, 60]
sigma2_values = obtain_sigma2_values(X0_test, mus)
sigma2_values = sigma2_values * (1+noise)
emittance_reconstructed, beta_reconstructed, alpha_reconstructed = reconstruct_Sigma(mus, sigma2_values)
relative_emittance_error, relative_beta_error, relative_alpha_error = relative_errors(emittance_reconstructed, beta_reconstructed, alpha_reconstructed)
print(f"Reconstructed emittance: {emittance_reconstructed}, initial emittance: {emittance}, relative emittance error: {relative_emittance_error} %")
print(f"Reconstructed beta: {beta_reconstructed}, initial emittance: {beta}, relative emittance error: {relative_beta_error} %")
print(f"Reconstructed alpha: {alpha_reconstructed}, initial emittance: {alpha}, relative emittance error: {relative_alpha_error} %")

Reconstructed emittance: 2.444876009686108, initial emittance: 2.0, relative emittance error: 22.24380048430541 %
Reconstructed beta: 2.4681539090793225, initial emittance: 3.0, relative emittance error: 17.72820303068925 %
Reconstructed alpha: -0.7563040590901401, initial emittance: -1.0, relative emittance error: 24.36959409098599 %


In [29]:
mus = [0, 20, 40]
sigma2_values = obtain_sigma2_values(X0_test, mus)
sigma2_values = sigma2_values * (1+noise)
emittance_reconstructed, beta_reconstructed, alpha_reconstructed = reconstruct_Sigma(mus, sigma2_values)
relative_emittance_error, relative_beta_error, relative_alpha_error = relative_errors(emittance_reconstructed, beta_reconstructed, alpha_reconstructed)
print(f"Reconstructed emittance: {emittance_reconstructed}, initial emittance: {emittance}, relative emittance error: {relative_emittance_error} %")
print(f"Reconstructed beta: {beta_reconstructed}, initial emittance: {beta}, relative emittance error: {relative_beta_error} %")
print(f"Reconstructed alpha: {alpha_reconstructed}, initial emittance: {alpha}, relative emittance error: {relative_alpha_error} %")

Reconstructed emittance: 3.057932669054088, initial emittance: 2.0, relative emittance error: 52.89663345270439 %
Reconstructed beta: 1.973336542556258, initial emittance: 3.0, relative emittance error: 34.22211524812474 %
Reconstructed alpha: -0.5698745289929741, initial emittance: -1.0, relative emittance error: 43.01254710070259 %


In [30]:
mus = [0, 5, 10]
sigma2_values = obtain_sigma2_values(X0_test, mus)
sigma2_values = sigma2_values * (1+noise)
emittance_reconstructed, beta_reconstructed, alpha_reconstructed = reconstruct_Sigma(mus, sigma2_values)
relative_emittance_error, relative_beta_error, relative_alpha_error = relative_errors(emittance_reconstructed, beta_reconstructed, alpha_reconstructed)
print(f"Reconstructed emittance: {emittance_reconstructed}, initial emittance: {emittance}, relative emittance error: {relative_emittance_error} %")
print(f"Reconstructed beta: {beta_reconstructed}, initial emittance: {beta}, relative emittance error: {relative_beta_error} %")
print(f"Reconstructed alpha: {alpha_reconstructed}, initial emittance: {alpha}, relative emittance error: {relative_alpha_error} %")

Reconstructed emittance: 9.240268288482644, initial emittance: 2.0, relative emittance error: 362.01341442413224 %
Reconstructed beta: 0.6530470860940693, initial emittance: 3.0, relative emittance error: 78.23176379686436 %
Reconstructed alpha: -0.10350139629199151, initial emittance: -1.0, relative emittance error: 89.64986037080085 %


In [31]:
phase_sets = [
    [0, 5, 10],
    [0, 10, 20],
    [0, 20, 40],
    [0, 30, 60],
    [0, 45, 90]
]

noise_level = 0.01
rng = np.random.default_rng(42)
phase_spans = []
relative_errors = []

for mus in phase_sets:
    # ideal beam sizes
    sigma2_values = np.array(obtain_sigma2_values(X0_test, mus))

    # add the same level of measurement noise
    noise = rng.normal(0, noise_level, size=3)
    sigma2_noisy = sigma2_values * (1 + noise)

    # reconstruct emittance
    emittance_rec, _, _ = reconstruct_Sigma(mus, sigma2_noisy)

    # relative error
    error = abs(emittance_rec - emittance) / emittance * 100
    phase_span = max(mus) - min(mus)
    phase_spans.append(phase_span)
    relative_errors.append(error)

In [32]:
plt.figure()
plt.plot(phase_spans, relative_errors)

plt.xlabel("Phase span [deg]")
plt.ylabel("Relative emittance error [%]")
plt.title("Effect of phase-space coverage on emittance reconstruction")

plt.grid()
plt.show()

# conclusions : small phase scan and a measurement error -> wrong reconstruction

In [33]:
# plot cond A vs relative emittance error
#
# phase coverage matters
#
#
# What phase coverage does CLEAR actually provide?
#
# Is that coverage good enough for a stable reconstruction?



Knowing that the reconstruction works for sufficient phase advance, now we try to test the same method with bunch B0 made from Twiss parameters at CLEAR.

In [34]:
I_CLEAR =  InterfaceCLEAR_RFTrack()
X0_CLEAR = I_CLEAR.B0.get_phase_space('%x %xp').T
mus = [0 , 45, 90]
sigma2_values = obtain_sigma2_values(X0_CLEAR, mus)
emittance_reconstructed, beta_reconstructed, alpha_reconstructed = reconstruct_Sigma(mus, sigma2_values)
_, _, beta_gamma = I_CLEAR.get_beam_factors()

emittance_reconstructed_normalized = emittance_reconstructed* beta_gamma
print(emittance_reconstructed_normalized)
print(beta_reconstructed)
print(alpha_reconstructed)

7.039296000000018
15.592257420175946
-0.4911664057045168


## Actual Beam Transport - CLEAR

## Reconstruction using the actual CLEAR transport

Instead of assuming an ideal phase-space rotation, we now use the actual
transport through the CLEAR lattice.

The horizontal phase space is transported according to

$$
\begin{pmatrix}
x_s \\
x'_s
\end{pmatrix}
=
\begin{pmatrix}
R_{11} & R_{12} \\
R_{21} & R_{22}
\end{pmatrix}
\begin{pmatrix}
x_0 \\
x'_0
\end{pmatrix}.
$$

Therefore,

$$
x_s = R_{11}x_0 + R_{12}x'_0.
$$

### Determination of $R_{11}$

To determine the first column of the transport matrix, we track a test particle with the initial coordinates

$$
\mathbf{e}_1 =
\begin{pmatrix}
1 \\
0
\end{pmatrix},
$$

which corresponds to

$$
x_0 = 1,
\qquad
x'_0 = 0.
$$

The particle is transported through the actual RF-Track lattice according to

$$
\begin{pmatrix}
x_s \\
x'_s
\end{pmatrix}
=
\begin{pmatrix}
R_{11} & R_{12} \\
R_{21} & R_{22}
\end{pmatrix}
\begin{pmatrix}
x_0 \\
x'_0
\end{pmatrix}.
$$

Substituting the initial coordinates of the test particle gives

$$
\begin{pmatrix}
x_s \\
x'_s
\end{pmatrix}
=
\begin{pmatrix}
R_{11} & R_{12} \\
R_{21} & R_{22}
\end{pmatrix}
\begin{pmatrix}
1 \\
0
\end{pmatrix}
=
\begin{pmatrix}
R_{11} \\
R_{21}
\end{pmatrix}.
$$

Therefore, the horizontal position of the test particle at the screen directly gives

$$
\boxed{R_{11} = x_s}.
$$

In [35]:
quadrupole = "CA.QDD0515"
screen = "CA.BTV0730"
quadrupole_element = I_CLEAR.lattice[quadrupole]
screen_element = I_CLEAR.lattice[screen]

lattice_view = rft.Lattice_view(I_CLEAR.lattice, quadrupole_element, screen_element)

In [36]:
e1_particle = np.array([[1.0, 0.0, 0.0, 0.0, 0.0, I_CLEAR.Pref]])
e1_bunch = rft.Bunch6d(rft.electronmass, 0.0, I_CLEAR.Q, e1_particle)
e1_at_screen = lattice_view.track(e1_bunch)
phase_space_e1 = e1_at_screen.get_phase_space("%x %xp")
R11 = phase_space_e1[0, 0]
R21 = phase_space_e1[0, 1]
print(R11)
print(R21)

3.209516167846554
0.28126837380875125


### Determination of $R_{12}$

To determine the second column of the transport matrix, we track a second test particle with the initial coordinates

$$
\mathbf{e}_2 =
\begin{pmatrix}
0 \\
1
\end{pmatrix},
$$

which corresponds to

$$
x_0 = 0,
\qquad
x'_0 = 1.
$$

The particle is transported through the actual RF-Track lattice according to

$$
\begin{pmatrix}
x_s \\
x'_s
\end{pmatrix}
=
\begin{pmatrix}
R_{11} & R_{12} \\
R_{21} & R_{22}
\end{pmatrix}
\begin{pmatrix}
x_0 \\
x'_0
\end{pmatrix}.
$$

Substituting the initial coordinates of the second test particle gives

$$
\begin{pmatrix}
x_s \\
x'_s
\end{pmatrix}
=
\begin{pmatrix}
R_{11} & R_{12} \\
R_{21} & R_{22}
\end{pmatrix}
\begin{pmatrix}
0 \\
1
\end{pmatrix}
=
\begin{pmatrix}
R_{12} \\
R_{22}
\end{pmatrix}.
$$

Therefore, the horizontal position of the second test particle at the screen directly gives

$$
\boxed{R_{12} = x_s}.
$$

In [37]:
e2_particle = np.array([[0.0, 1.0, 0.0, 0.0, 0.0, I_CLEAR.Pref]])
e2_bunch = rft.Bunch6d(rft.electronmass, 0.0, I_CLEAR.Q, e2_particle)
e2_at_screen = lattice_view.track(e2_bunch)
phase_space_e2 = e2_at_screen.get_phase_space("%x %xp")
R12 = phase_space_e2[0, 0]
R22 = phase_space_e2[0, 1]
print(R12)
print(R22)

6.014432899281646
0.838652812510615


In [38]:
M = np.array([
    [R11, R12],
    [R21, R22]
])

print(M)

[[3.20951617 6.0144329 ]
 [0.28126837 0.83865281]]


In [39]:
quadrupole_data = I_CLEAR.get_quadrupoles([quadrupole])
bdes = np.asarray(quadrupole_data.get("bdes", []), dtype=float)
K1L_0 = float(bdes[0]) # -0.8020069053131611
deltas = np.linspace(-0.50, 0.50, 5)
K1L_values = K1L_0 * (1 + deltas)

start_element = I_CLEAR.lattice[I_CLEAR.start]
quad_index = I_CLEAR.sequence.index(quadrupole)
before_quad_name = I_CLEAR.sequence[quad_index - 1]
before_quad = I_CLEAR.lattice[before_quad_name]
upstream_view = rft.Lattice_view(I_CLEAR.lattice, start_element, before_quad)

B_ref = upstream_view.track(I_CLEAR.B0.displaced(0, 0, 0, 0, 0, 0, 0))

sigma2_values = []
R11_values = []
R21_values = []
R12_values = []
R22_values = []
for K1L in K1L_values:
    I_CLEAR.set_quadrupoles([quadrupole],[K1L])
    lattice_view = rft.Lattice_view(I_CLEAR.lattice, I_CLEAR.lattice[quadrupole], I_CLEAR.lattice[screen])
    B_screen = lattice_view.track(B_ref)
    phase_space_at_screen = B_screen.get_phase_space('%x %xp')
    sigma2_values.append(np.var(phase_space_at_screen[:,0]))

    e1_particle = np.array([[1.0, 0.0, 0.0, 0.0, 0.0, I_CLEAR.Pref]])
    e1_bunch = rft.Bunch6d(rft.electronmass, 0.0, I_CLEAR.Q, e1_particle)
    e1_at_screen = lattice_view.track(e1_bunch)
    phase_space_e1 = e1_at_screen.get_phase_space("%x %xp")
    R11 = phase_space_e1[0, 0]
    R21 = phase_space_e1[0, 1]
    R11_values.append(R11)
    R21_values.append(R21)

    e2_particle = np.array([[0.0, 1.0, 0.0, 0.0, 0.0, I_CLEAR.Pref]])
    e2_bunch = rft.Bunch6d(rft.electronmass, 0.0, I_CLEAR.Q, e2_particle)
    e2_at_screen = lattice_view.track(e2_bunch)
    phase_space_e2 = e2_at_screen.get_phase_space("%x %xp")
    R12 = phase_space_e2[0, 0]
    R22 = phase_space_e2[0, 1]
    R12_values.append(R12)
    R22_values.append(R22)

I_CLEAR.set_quadrupoles([quadrupole], [K1L_0])


Previously, for the ideal phase-space rotation, we assumed

$$
R_{11,i} = \cos\mu_i,
\qquad
R_{12,i} = \sin\mu_i.
$$

Therefore, each row of the reconstruction matrix was

$$
A_i =
\begin{pmatrix}
\cos^2\mu_i &
2\cos\mu_i\sin\mu_i &
\sin^2\mu_i
\end{pmatrix}.
$$

For the actual CLEAR lattice, we no longer assume an ideal rotation.
Therefore:

$$
A_i =
\begin{pmatrix}
R_{11,i}^2 &
2R_{11,i}R_{12,i} &
R_{12,i}^2
\end{pmatrix}.
$$

The corresponding beam-size equation is the same:

$$
\sigma_{x,i}^2 =
R_{11,i}^2\Sigma_{11}
+
2R_{11,i}R_{12,i}\Sigma_{12}
+
R_{12,i}^2\Sigma_{22}.
$$

In [40]:
R11_values = np.asarray(R11_values)
R12_values = np.asarray(R12_values)
sigma2_values = np.asarray(sigma2_values)

A = np.column_stack([
    R11_values**2,
    2*R11_values*R12_values,
    R12_values**2,
])

print(A)
print(A.shape) # 5 measurements, 3 unknowns - Sigma11, Sigma12, Sigma22

[[ 0.69990256  9.62050044 33.05961214]
 [ 4.05696886 23.69330892 34.59312275]
 [10.30099403 38.60683926 36.1734031 ]
 [19.5602327  54.38390606 37.80130436]
 [31.96601884 71.04771756 39.47768875]]
(5, 3)


In [41]:
Sigma_elements = np.linalg.lstsq(A, sigma2_values)[0]
Sigma11 = Sigma_elements[0]
Sigma12 = Sigma_elements[1]
Sigma22 = Sigma_elements[2]

Sigma_fit = np.array([
    [Sigma11, Sigma12],
    [Sigma12, Sigma22]
])

emittance_fit = np.sqrt(np.linalg.det(Sigma_fit)) * beta_gamma
beta_fit = Sigma11 / emittance_fit
alpha_fit = -Sigma12 / emittance_fit

print(emittance_fit)
print(beta_fit)
print(alpha_fit)

7.0392959999034845
0.40769958158179337
-0.05986832963887969


Now, we infer phase space from RF_Track, which in reality, would not be possible by screens to see, to check if the fit was successful.

In [42]:
phase_space_ref = B_ref.get_phase_space("%x %xp")
Sigma_true = np.cov(phase_space_ref[:,0], phase_space_ref[:,1]) # first column, second column
emittance_true = np.sqrt(np.linalg.det(Sigma_true)) * beta_gamma
beta_true = Sigma_true[0][0]/emittance_true
alpha_true = -Sigma_true[0][1]/emittance_true

print(emittance_true)
print(beta_true)
print(alpha_true)

7.040000000003188
0.4076995815760203
-0.0598683296380311


## Are CLEAR screens in good/sufficient phase advances with respect to each other?

In [43]:
def transport_matrix_x(start, end):
    e1_e2_particles = np.array([
        [1.0, 0.0, 0.0, 0.0, 0.0, I_nom.Pref],
        [0.0, 1.0, 0.0, 0.0, 0.0, I_nom.Pref],
    ])
    bunch_ref = rft.Bunch6d(rft.electronmass, 0.0, I_nom.Q, e1_e2_particles)
    lattice_view = rft.Lattice_view(I_nom.lattice, I_nom.lattice[start], I_nom.lattice[end])
    e1_e2_at_screen = lattice_view.track(bunch_ref)
    phase_space = e1_e2_at_screen.get_phase_space("%x %xp")

    return np.array([
        [phase_space[0, 0], phase_space[1, 0]],
        [phase_space[0, 1], phase_space[1, 1]],
    ])

## Nominal CLEAR: phase advance between screens

The horizontal phase-space transport from screen \(i\) to screen \(j\) is

$$
\begin{pmatrix}
x_j \\
x'_j
\end{pmatrix}
=
\begin{pmatrix}
R_{11} & R_{12} \\
R_{21} & R_{22}
\end{pmatrix}
\begin{pmatrix}
x_i \\
x'_i
\end{pmatrix},
\qquad
$$

The Twiss parameter \(\beta_j\) at screen \(j\) is propagated as

$$
\beta_j =
R_{11}^{2}\beta_i
- 2R_{11}R_{12}\alpha_i
+ R_{12}^{2}\gamma_i,
$$

where

$$
\gamma_i =
\frac{1+\alpha_i^2}{\beta_i}.
$$

The phase advance between screens \(i\) and \(j\) is obtained from

$$
\sin(\Delta\mu_{i \to j}) =
\frac{R_{12}}{\sqrt{\beta_i\beta_j}},
$$

and

$$
\cos(\Delta\mu_{i \to j}) =
R_{11}\sqrt{\frac{\beta_i}{\beta_j}}
- \alpha_i\sin(\Delta\mu_{i \to j}).
$$

Finally,

$$
\Delta\mu_{i \to j}
=
\operatorname{atan2}
\left(
\sin(\Delta\mu_{i \to j}),
\cos(\Delta\mu_{i \to j})
\right).
$$

In [44]:
I_nom = InterfaceCLEAR_RFTrack()

screens = [
    "CA.BTV0215",
    "CA.BTV0390",
    "CA.BTV0620",
    "CA.BTV0730",
    "CA.BTV0805",
    "CA.BTV0810",
    "CA.BTV0910",
]
reference_screen = screens[0]

I_nom.lattice.track(I_nom.B0.displaced(0, 0, 0, 0, 0, 0, 0))
B_ref_screen = I_nom.lattice[reference_screen].get_bunch()
phase_space_ref = B_ref_screen.get_phase_space("%x %xp")
Sigma_ref = np.cov(phase_space_ref, rowvar=False, bias=True)

emittance_ref_geom = np.sqrt(np.linalg.det(Sigma_ref))
beta_ref = Sigma_ref[0, 0] / emittance_ref_geom
alpha_ref = -Sigma_ref[0, 1] / emittance_ref_geom
gamma_ref = (1 + alpha_ref**2) / beta_ref
mu_values = [0.0]

for screen in screens[1:]:
    M = transport_matrix_x(reference_screen, screen)
    beta_screen = (M[0, 0]**2 * beta_ref - 2 * M[0, 0] * M[0, 1] * alpha_ref + M[0, 1]**2 * gamma_ref)
    sin_mu = M[0, 1] / np.sqrt(beta_ref * beta_screen)
    cos_mu = M[0, 0] * np.sqrt(beta_ref / beta_screen) - alpha_ref * sin_mu
    mu_values.append(np.rad2deg(np.arctan2(sin_mu, cos_mu)) % 360)

mu_values = np.rad2deg(np.unwrap(np.deg2rad(mu_values)))
x_positions = np.arange(len(screens))

plt.figure(figsize=(8, 4))
plt.plot(x_positions, mu_values, "o-")
for x, mu in zip(x_positions, mu_values):
    plt.annotate(f"{mu:.1f}°", (x, mu), xytext=(0, 8), textcoords="offset points", ha="center")
plt.xticks(x_positions, screens, rotation=35, ha="right")
plt.ylabel(r"Phase advance from first screen [$^\circ$]")
plt.title("Nominal CLEAR horizontal phase advance")
plt.grid(True)
plt.tight_layout()
plt.show()


Optimization?

In [45]:
from scipy.optimize import least_squares

sigma2_measurements = sigma2_values.copy()

def predict_sigma2(parameters):
    emit, beta, alpha = parameters
    gamma = (1 + alpha**2) / beta
    Sigma_elements = [emit * beta, -emit * alpha, emit * gamma]
    return A @ Sigma_elements

def merit_residuals(parameters):
    emit = np.exp(parameters[0])
    beta = np.exp(parameters[1])
    alpha = parameters[2]
    return (predict_sigma2([emit, beta, alpha]) - sigma2_measurements) / sigma2_measurements

fit = least_squares(merit_residuals, x0=[np.log(0.01), np.log(100.0), 0.0], method="trf", x_scale="jac", max_nfev=5000)
emit_fit = np.exp(fit.x[0])# > 0
beta_fit = np.exp(fit.x[1]) # > 0
alpha_fit = fit.x[2]

phase_space_true = B_ref.get_phase_space("%x %xp")
Sigma_true = np.cov(phase_space_true, rowvar=False, bias=True)
emit_true = np.sqrt(np.linalg.det(Sigma_true))
beta_true = Sigma_true[0, 0] / emit_true
alpha_true = -Sigma_true[0, 1] / emit_true


print(f"cond(A) = {np.linalg.cond(A):.2f}")
print(f"merit at minimum = {np.sum(fit.fun**2):.2e}")
print(f"true: emit={emit_true:.6g}, beta={beta_true:.6g}, alpha={alpha_true:.6g}")
print(f"fit:  emit={emit_fit:.6g}, beta={beta_fit:.6g}, alpha={alpha_fit:.6g}")

plt.plot(K1L_values, sigma2_measurements, "o", label="RF-Track scan")
plt.plot(K1L_values, predict_sigma2([emit_fit, beta_fit, alpha_fit]), "-", label="merit fit")
plt.xlabel("QDD0515 K1L")
plt.ylabel(r"$\sigma_x^2$")
plt.legend()
plt.show()


cond(A) = 31.68
merit at minimum = 7.96e-16
true: emit=0.018167, beta=157.974, alpha=-23.1976
fit:  emit=0.0181719, beta=157.931, alpha=-23.1913


## Phase advance created by the quadrupole scan

In [46]:
phase_space_ref = B_ref.get_phase_space("%x %xp")
Sigma_ref = np.cov(phase_space_ref, rowvar=False, bias=True)
emit_ref = np.sqrt(np.linalg.det(Sigma_ref))
beta_ref = Sigma_ref[0, 0] / emit_ref
alpha_ref = -Sigma_ref[0, 1] / emit_ref
gamma_ref = (1 + alpha_ref**2) / beta_ref

phase_scan_deg = []
for R11, R12 in zip(R11_values, R12_values):
    beta_screen = R11**2 * beta_ref - 2 * R11 * R12 * alpha_ref + R12**2 * gamma_ref
    sin_mu = R12 / np.sqrt(beta_ref * beta_screen)
    cos_mu = R11 * np.sqrt(beta_ref / beta_screen) - alpha_ref * sin_mu
    phase_scan_deg.append(np.rad2deg(np.arctan2(sin_mu, cos_mu)))

phase_scan_deg = np.rad2deg(np.unwrap(np.deg2rad(phase_scan_deg)))
for K1L, mu in zip(K1L_values, phase_scan_deg):
    print(f"K1L = {K1L:.4f}, phase advance = {mu:.2f} deg")

plt.plot(K1L_values, phase_scan_deg, "o-")
plt.xlabel("QDD0515 K1L")
plt.ylabel(r"$\Delta\mu$ to BTV0730 [deg]")
plt.title("Phase advance sampled by the QDD0515 scan")
plt.grid()
plt.show()


K1L = -0.4010, phase advance = 1.24 deg
K1L = -0.6015, phase advance = 0.74 deg
K1L = -0.8020, phase advance = 0.53 deg
K1L = -1.0025, phase advance = 0.42 deg
K1L = -1.2030, phase advance = 0.35 deg
